In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def clip_y(y_hat, epsilon=1e-8):
    return np.clip(y_hat, epsilon, 1 - epsilon)

def binary_cross_entropy(y_true, y_hat):
    y_hat = clip_y(y_hat)
    return -np.mean(y_true * np.log(y_hat) + (1 - y_true) * np.log(1 - y_hat))

def accuracy(y_true, y_hat):
    y_pred_bin = (y_hat >= 0.5).astype(int)
    return np.mean(y_true == y_pred_bin)

def get_numeric_data(N=3000):
    X = np.random.uniform(-2, 2, (N, 2))
    y = (np.sum(X**2, axis=1) > 1.5).astype(int).reshape(-1, 1)
    return split_data(X, y)

def get_image_data(N=3000):
    X = np.zeros((N, 8, 8))
    y = np.zeros((N, 1))
    for i in range(N):
        if np.random.rand() > 0.5:
            X[i, :, 3:5] = 1.0
            y[i] = 0
        else:
            X[i, 3:5, :] = 1.0
            y[i] = 1
    X += np.random.normal(0, 0.1, X.shape)
    X = X.reshape(N, 1, 8, 8)
    return split_data(X, y)

def split_data(X, y):
    N = X.shape[0]
    indices = np.random.permutation(N)
    train_idx, val_idx = int(0.7*N), int(0.85*N)
    return (X[indices[:train_idx]], y[indices[:train_idx]]), \
           (X[indices[train_idx:val_idx]], y[indices[train_idx:val_idx]]), \
           (X[indices[val_idx:]], y[indices[val_idx:]])

class Dense:
    def __init__(self, n_in, n_out):
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0/n_in)
        self.b = np.zeros((1, n_out))
        self.dW, self.db = None, None
        self.vW, self.vb = np.zeros_like(self.W), np.zeros_like(self.b)
        self.mW, self.mb = np.zeros_like(self.W), np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, grad_output):
        m = self.x.shape[0]
        self.dW = np.dot(self.x.T, grad_output)
        self.db = np.sum(grad_output, axis=0, keepdims=True)
        return np.dot(grad_output, self.W.T)

class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size=3):
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.1
        self.b = np.zeros((out_channels, 1))
        self.vW, self.vb = np.zeros_like(self.W), np.zeros_like(self.b)
        self.mW, self.mb = np.zeros_like(self.W), np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        n, c, h, w = x.shape
        f, _, kh, kw = self.W.shape
        out_h, out_w = h - kh + 1, w - kw + 1
        out = np.zeros((n, f, out_h, out_w))
        for i in range(out_h):
            for j in range(out_w):
                x_slice = x[:, :, i:i+kh, j:j+kw]
                out[:, :, i, j] = np.sum(x_slice[:, np.newaxis, :, :, :] * self.W, axis=(2, 3, 4)) + self.b.T
        return out

    def backward(self, grad_output):
        n, f, out_h, out_w = grad_output.shape
        n, c, h, w = self.x.shape
        f, _, kh, kw = self.W.shape
        self.dW = np.zeros_like(self.W)
        self.db = np.sum(grad_output, axis=(0, 2, 3)).reshape(-1, 1)
        grad_input = np.zeros_like(self.x)
        for i in range(out_h):
            for j in range(out_w):
                x_slice = self.x[:, :, i:i+kh, j:j+kw]
                for k in range(f):
                    self.dW[k] += np.sum(x_slice * grad_output[:, k, i, j][:, np.newaxis, np.newaxis, np.newaxis], axis=0)
                for k in range(f):
                    grad_input[:, :, i:i+kh, j:j+kw] += self.W[k] * grad_output[:, k, i, j][:, np.newaxis, np.newaxis, np.newaxis]
        return grad_input

class MaxPooling:
    def __init__(self, size=2):
        self.size = size

    def forward(self, x):
        self.x = x
        n, c, h, w = x.shape
        out = x.reshape(n, c, h//self.size, self.size, w//self.size, self.size).max(axis=(3, 5))
        return out

    def backward(self, grad_output):
        n, c, oh, ow = grad_output.shape
        s = self.size
        grad_input = np.zeros_like(self.x)
        for i in range(oh):
            for j in range(ow):
                patch = self.x[:, :, i*s:(i+1)*s, j*s:(j+1)*s]
                mask = (patch == patch.max(axis=(2, 3), keepdims=True))
                grad_input[:, :, i*s:(i+1)*s, j*s:(j+1)*s] = mask * grad_output[:, :, i, j][:, :, np.newaxis, np.newaxis]
        return grad_input

class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)
    def backward(self, grad_output):
        return grad_output * (self.x > 0)

class Sigmoid:
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        return self.out
    def backward(self, grad_output):
        return grad_output * (self.out * (1 - self.out))

class Flatten:
    def forward(self, x):
        self.shape = x.shape
        return x.reshape(x.shape[0], -1)
    def backward(self, grad_output):
        return grad_output.reshape(self.shape)

class Optimizer:
    def __init__(self, layers, lr=0.01, beta=0.9, method='sgd'):
        self.layers = [l for l in layers if hasattr(l, 'W')]
        self.lr, self.beta, self.method = lr, beta, method
        self.t = 0

    def step(self):
        self.t += 1
        for l in self.layers:
            if self.method == 'sgd':
                l.W -= self.lr * l.dW
                l.b -= self.lr * l.db
            elif self.method == 'momentum':
                l.vW = self.beta * l.vW + self.lr * l.dW
                l.vb = self.beta * l.vb + self.lr * l.db
                l.W -= l.vW
                l.b -= l.vb
            elif self.method == 'adam':
                b1, b2, eps = 0.9, 0.999, 1e-8
                l.mW = b1 * l.mW + (1-b1) * l.dW
                l.mb = b1 * l.mb + (1-b1) * l.db
                l.vW = b2 * l.vW + (1-b2) * (l.dW**2)
                l.vb = b2 * l.vb + (1-b2) * (l.db**2)
                mhatW = l.mW / (1 - b1**self.t)
                mhatb = l.mb / (1 - b1**self.t)
                vhatW = l.vW / (1 - b2**self.t)
                vhatb = l.vb / (1 - b2**self.t)
                l.W -= self.lr * mhatW / (np.sqrt(vhatW) + eps)
                l.b -= self.lr * mhatb / (np.sqrt(vhatb) + eps)

class Model:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def train(self, train_data, val_data, epochs=100, lr=0.01, opt_type='sgd'):
        X_train, y_train = train_data
        X_val, y_val = val_data
        optimizer = Optimizer(self.layers, lr=lr, method=opt_type)
        history = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

        for epoch in range(epochs):
            y_hat = self.forward(X_train)
            loss = binary_cross_entropy(y_train, y_hat)
            grad = (clip_y(y_hat) - y_train) / len(y_train)
            self.backward(grad)
            optimizer.step()

            val_hat = self.forward(X_val)
            val_loss = binary_cross_entropy(y_val, val_hat)
            history['loss'].append(loss)
            history['val_loss'].append(val_loss)
            history['acc'].append(accuracy(y_train, y_hat))
            history['val_acc'].append(accuracy(y_val, val_hat))

            if epoch % 20 == 0:
                print(f"Epoch {epoch}: Loss={loss:.4f}, Val Acc={history['val_acc'][-1]:.4f}")
        return history

def run_part1():
    train, val, test = get_numeric_data()
    layers = [
        Dense(2, 8), ReLU(),
        Dense(8, 8), ReLU(),
        Dense(8, 8), ReLU(),
        Dense(8, 8), ReLU(),
        Dense(8, 1), Sigmoid()
    ]
    model = Model(layers)
    history = model.train(train, val, epochs=200, opt_type='momentum')

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.legend()
    plt.show()

def run_part2():
    train, val, test = get_image_data()
    layers = [
        Conv2D(1, 4, kernel_size=3), ReLU(),
        MaxPooling(2),
        Flatten(),
        Dense(4 * 3 * 3, 1), Sigmoid()
    ]
    model = Model(layers)
    history = model.train(train, val, epochs=50, lr=0.05, opt_type='adam')

    test_hat = model.forward(test[0])
    print(f"Final Test Accuracy: {accuracy(test[1], test_hat):.4f}")

if __name__ == "__main__":
    run_part1()
    run_part2()